In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/update-ds-semeval2026/subtask2/test/tel.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/mya.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/amh.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/eng.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/ben.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/hau.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/khm.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/pan.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/pol.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/arb.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/hin.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/nep.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/ita.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/zho.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/urd.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/rus.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/swa.csv
/kaggle/input/

In [2]:
import re
def clean_text(s: str) -> str:
    s = re.sub(r"http\S+|www\S+", "<URL>", str(s))
    s = re.sub(r"@\w+", "<USER>", s)
    return s.strip()

In [3]:
df_1 = pd.read_csv("/kaggle/input/augdatasetusingst1/augmented_data_label0.csv")
df_2 = pd.read_csv("/kaggle/input/augdatasetusingst1/augmented_data_label1.csv")
df_original = pd.read_csv("/kaggle/input/update-ds-semeval2026/subtask1/train/eng.csv")
df_test = pd.read_csv("/kaggle/input/update-ds-semeval2026/subtask1/test/eng.csv")
df_dev = pd.read_csv("/kaggle/input/update-ds-semeval2026/subtask1/dev/eng.csv")

In [4]:
df_1.shape

(1920, 2)

In [5]:
df_2.shape

(1116, 2)

In [6]:
# import thư viện
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

2026-02-05 19:06:30.128776: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770318390.309090      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770318390.359188      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770318390.806949      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770318390.806983      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770318390.806986      24 computation_placer.cc:177] computation placer alr

In [7]:
# df_2_sample = df_2.sample(n=850, random_state=42)
df_train = pd.concat([df_1,df_2 ,df_original], ignore_index=True)

# df_train = df_original
df_train['text'] = df_train['text'].apply(clean_text)

# tạo tập huấn luyện và tập kiểm tra
train, val = train_test_split(df_train, test_size=0.2, random_state=42, stratify=df_train['polarization'])

In [8]:
df_train["polarization"].value_counts()

polarization
0    3967
1    2291
Name: count, dtype: int64

In [9]:
# config model 
MODEL_NAME = "roberta-base"
NUM_LABELS = 2
MAX_LEN = 128
LR = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 7
TRAIN_BS = 16
GRAD_ACCUM = 2          
EVAL_BS = 24
WARMUP_RATIO = 0.06
SEED = 42
OUTPUT_DIR = "./model_output/"

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# tạo dataset class
class PolarizationDataset(Dataset):
    def __init__(self, df, tokenizer, max_len: int, is_test: bool = True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['text']
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if not self.is_test:
            label = row['polarization']
            item['labels'] = torch.tensor(label, dtype=torch.long)
        return item


# khởi tạo tokenizer và dataset
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = PolarizationDataset(train, tokenizer, MAX_LEN, is_test=False)
val_dataset = PolarizationDataset(val, tokenizer, MAX_LEN, is_test=False)
data_collator = DataCollatorWithPadding(tokenizer)


# Use class weights
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df_train['polarization']),
    y=df_train['polarization'].values
)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

class WeightedRoberta(nn.Module):
    def __init__(self, model_name, num_labels, weights):
        super().__init__()
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, 
            num_labels=num_labels,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1
        )
        self.loss_fn = nn.CrossEntropyLoss(weight=weights)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        # FIX: Loại bỏ 'num_items_in_batch' khỏi kwargs trước khi đưa vào RoBERTa
        kwargs.pop("num_items_in_batch", None)
        
        outputs = self.model(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            **kwargs
        )
        
        logits = outputs.logits
        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)
        
        return SequenceClassifierOutput(loss=loss, logits=logits)
model = WeightedRoberta(MODEL_NAME, NUM_LABELS, class_weights).to(device)

# Khởi tạo model
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME, 
#     num_labels=NUM_LABELS,
#     hidden_dropout_prob=0.2,         
#     attention_probs_dropout_prob=0.2
# ).to(device)

# định nghĩa hàm tính metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'f1_macro': f1_score(labels, preds, average='macro'), 
        'precision': precision_score(labels, preds, average='weighted', zero_division=0),
        'recall': recall_score(labels, preds, average='weighted', zero_division=0)
    }

# Thiết lập tham số huấn luyện
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",     
    greater_is_better=True,
    fp16=torch.cuda.is_available(),        
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_24/3132309003.py:137: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
# huấn luyện mô hình
trainer.train()

# lưu mô hình
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Epoch,Training Loss,Validation Loss,F1 Weighted,F1 Macro,Precision,Recall
1,0.637500,0.275414,0.878953,0.871018,0.882860,0.877796
2,0.446300,0.263856,0.894098,0.887336,0.898874,0.892971
3,0.299200,0.320152,0.869586,0.862946,0.885162,0.867412
4,0.182600,0.477452,0.900090,0.892261,0.900031,0.900160
5,0.153200,0.617051,0.883906,0.873847,0.884423,0.884984
6,0.131400,0.625505,0.892482,0.884484,0.892993,0.892173
7,0.093300,0.682889,0.894937,0.887200,0.895611,0.894569


('./model_output/tokenizer_config.json',
 './model_output/special_tokens_map.json',
 './model_output/vocab.json',
 './model_output/merges.txt',
 './model_output/added_tokens.json',
 './model_output/tokenizer.json')

In [11]:
print("Best metric:", trainer.state.best_metric)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

Best metric: 0.892260889739175
Best checkpoint: ./model_output/checkpoint-628


In [12]:
# model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
# tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
# model.to(device)
# model.eval()

# Lưu toàn bộ trọng số của lớp wrapper WeightedRoberta
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "weighted_roberta.pt"))

# Tokenizer thì vẫn lưu bằng cách cũ vì nó đã chạy đúng
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Đã lưu trọng số tại: {OUTPUT_DIR}/weighted_roberta.pt")

# 1. Khởi tạo khung model trống (phải có class WeightedRoberta trong code)
model = WeightedRoberta(MODEL_NAME, NUM_LABELS, class_weights)

# 2. Đổ trọng số đã lưu vào khung
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/weighted_roberta.pt", map_location=device))
model.to(device)
model.eval()

✅ Đã lưu trọng số tại: ./model_output//weighted_roberta.pt


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


WeightedRoberta(
  (model): RobertaForSequenceClassification(
    (roberta): RobertaModel(
      (embeddings): RobertaEmbeddings(
        (word_embeddings): Embedding(50265, 768, padding_idx=1)
        (position_embeddings): Embedding(514, 768, padding_idx=1)
        (token_type_embeddings): Embedding(1, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): RobertaEncoder(
        (layer): ModuleList(
          (0-11): 12 x RobertaLayer(
            (attention): RobertaAttention(
              (self): RobertaSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): RobertaSelfOutput(
                (de

In [13]:
import torch
import numpy as np

# ====== TEST SAMPLES (POLARIZATION) ======
test_texts = [
    "Anyone still defending those blue states is completely delusional at this point.",
    "Red state voters don’t care about facts, only blind loyalty to their side.",
    "Typical liberals always play the victim while ruining everything they touch.",
    "Conservatives pretend to love freedom but hate anyone who thinks differently.",
    "If you support that party, you’re part of the problem, no excuses.",
    "Blue states think they’re morally superior but can’t run anything properly.",
    "Red state politics are just ignorance wrapped in fake patriotism.",
    "Liberals live in a bubble and look down on everyone outside it.",
    "Conservatives only care about themselves, not society as a whole.",
    "This is exactly why people from the other side can’t be trusted."
]

# ====== INFERENCE LOOP ======
model.eval()

for idx, text in enumerate(test_texts, 1):
    # Tokenize
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=192,
        return_tensors="pt"
    ).to(device)

    # Inference
    with torch.inference_mode():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        pred_label = np.argmax(probs)

    # Print result
    print("=" * 80)
    print(f"🧾 [{idx}] Text: {text}")
    print(f"🔹 Probabilities: {probs}")
    print(f"🔸 Predicted label: {pred_label}")


🧾 [1] Text: Anyone still defending those blue states is completely delusional at this point.
🔹 Probabilities: [0.00199129 0.9980088 ]
🔸 Predicted label: 1
🧾 [2] Text: Red state voters don’t care about facts, only blind loyalty to their side.
🔹 Probabilities: [0.00149397 0.99850607]
🔸 Predicted label: 1
🧾 [3] Text: Typical liberals always play the victim while ruining everything they touch.
🔹 Probabilities: [0.00150887 0.99849117]
🔸 Predicted label: 1
🧾 [4] Text: Conservatives pretend to love freedom but hate anyone who thinks differently.
🔹 Probabilities: [0.00148301 0.998517  ]
🔸 Predicted label: 1
🧾 [5] Text: If you support that party, you’re part of the problem, no excuses.
🔹 Probabilities: [0.00241931 0.99758065]
🔸 Predicted label: 1
🧾 [6] Text: Blue states think they’re morally superior but can’t run anything properly.
🔹 Probabilities: [0.00448009 0.9955199 ]
🔸 Predicted label: 1
🧾 [7] Text: Red state politics are just ignorance wrapped in fake patriotism.
🔹 Probabilities: [0.0014

In [14]:
df_dev['text'] = df_dev['text'].apply(clean_text)
df_test['text'] = df_test['text'].apply(clean_text)

In [15]:
# 1. Đánh giá trên tập dev (đã có nhãn gold)
print("=== Evaluating on Dev set (Gold Labels) ===")

# Dùng dev_dataset đã được clean_text
dev_dataset = PolarizationDataset(df_dev, tokenizer, MAX_LEN, is_test=False)
dev_output = trainer.predict(dev_dataset)

# Lấy nhãn dự đoán và nhãn thực tế
dev_labels = dev_output.label_ids
dev_preds = np.argmax(dev_output.predictions, axis=1)

# Tính toán các chỉ số
dev_f1_macro = f1_score(dev_labels, dev_preds, average='macro')
dev_precision = precision_score(dev_labels, dev_preds, average='weighted', zero_division=0)
dev_recall = recall_score(dev_labels, dev_preds, average='weighted', zero_division=0)
dev_accuracy = accuracy_score(dev_labels, dev_preds)

print(f"\nDev Metrics Result:")
print(f" - F1 Macro: {dev_f1_macro:.4f}")
print(f" - Precision: {dev_precision:.4f}")
print(f" - Recall: {dev_recall:.4f}")
print(f" - Accuracy: {dev_accuracy:.4f}")

# Kiểm tra Confusion Matrix để xem model có bị bias nhãn không
print("\nConfusion Matrix:")
cm = confusion_matrix(dev_labels, dev_preds)
print(cm)

# ====== Save Dev results ======
submission_dev = pd.DataFrame({
    "id": df_dev["id"],
    "polarization": dev_preds,
})
submission_dev.to_csv(os.path.join(OUTPUT_DIR, "pred_eng_dev.csv"), index=False)
print(f"\n✅ Đã xuất file kết quả Dev tại: {os.path.join(OUTPUT_DIR, 'pred_eng_dev.csv')}")

=== Evaluating on Dev set (Gold Labels) ===



Dev Metrics Result:
 - F1 Macro: 0.8077
 - Precision: 0.8232
 - Recall: 0.8250
 - Accuracy: 0.8250

Confusion Matrix:
[[90 11]
 [17 42]]

✅ Đã xuất file kết quả Dev tại: ./model_output/pred_eng_dev.csv


In [16]:
# 2. Dự đoán trên tập test (để submit)
print("\n=== Predicting on Test set for submission ===")
test_dataset = PolarizationDataset(df_test, tokenizer, MAX_LEN, is_test=True) # is_test=True vì chưa có nhãn
test_output = trainer.predict(test_dataset)

test_logits = test_output.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_preds = np.argmax(test_probs, axis=1)

# ====== Save results ======
submission = pd.DataFrame({
    "id": df_test["id"],
    "polarization": test_preds,
})
submission.to_csv(os.path.join(OUTPUT_DIR, "pred_eng.csv"), index=False)

submission_probs = pd.DataFrame({
    "id": df_test["id"],
    "prob_class1": test_probs[:, 1],
})
submission_probs.to_csv(os.path.join(OUTPUT_DIR, "pred_eng_probs.csv"), index=False)

print(f"Saved submission file to: {os.path.join(OUTPUT_DIR, 'pred_eng.csv')}")


=== Predicting on Test set for submission ===


Saved submission file to: ./model_output/pred_eng.csv
